<a href="https://colab.research.google.com/github/GabAsencios/FocusGuard/blob/main/FocusGuard_CamDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Camera Detection Component: YOLOV8 + COCO8 Dataset**

In this notebook i will train a YoloV8 model with COCO8 datasets for phone and person detection.

In [ ]:
# Setup
!pip install ultralytics
import os
from ultralytics import YOLO

In [ ]:
def prepare_webcam_model(model_variant='yolov8n.pt'):
    """
    Loads a pretrained YOLOv8 model and prepares it for export.

    Args:
        model_variant (str): The YOLOv8 model version (n, s, m, l, or x).

    Returns:
        YOLO: The loaded ultralytics YOLO model.

    Example:
        model = prepare_webcam_model('yolov8n.pt')
    """
    # Load pretrained COCO weights as per proposal
    model_cam = YOLO(model_variant)

    # Save/Export the model
    model_cam.export(format='torchscript')
    return model

In [ ]:
# 3. Prepare Webcam Component
model_cam = prepare_webcam_model()

Ultralytics 8.4.38  Python-3.9.13 torch-2.6.0+cu124 CPU (AMD Ryzen 7 1700 Eight-Core Processor)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

TorchScript: starting export with torch 2.6.0+cu124...
TorchScript: export success  2.7s, saved as 'yolov8n.torchscript' (12.5 MB)

Export complete (3.1s)
Results saved to C:\Users\angel
Predict:         yolo predict task=detect model=yolov8n.torchscript imgsz=640 
Validate:        yolo val task=detect model=yolov8n.torchscript imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


In [ ]:
# # Download the weights (yolov8n.pt) for local inference
# from google.colab import files
# files.download('yolov8n.pt')

### 4. Evaluate the model with testing metrics

To evaluate the model, we can use the `model.val()` method. This will run the validation process on a specified dataset (COCO in this case) and output various performance metrics.

In [ ]:
# Evaluate the model's performance on the COCO dataset for specific classes
# Class IDs: 0 for person, 67 for cell phone in COCO dataset
metrics = model_cam.val(
    data='coco.yaml',
    classes=[0, 67]
)

# Print some key metrics
print(f"Map50-95: {metrics.box.map}")
print(f"Map50: {metrics.box.map50}")
print(f"Map75: {metrics.box.map75}")

Ultralytics 8.4.38  Python-3.9.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2070, 8192MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 603.258.7 MB/s, size: 184.5 KB)
val: Scanning C:\Users\angel\datasets\coco\labels\val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 4.7it/s 1:06
                   all       5000      11039      0.675      0.504      0.569      0.393
                person       2693      10777      0.774      0.661      0.742      0.508
            cell phone        214        262      0.575      0.346      0.396      0.277
Speed: 1.2ms preprocess, 3.0ms inference, 0.0ms loss, 1.9ms postprocess per image
Saving C:\Users\angel\runs\detect\val5\predictions.json...

Evaluating faster-coco-eval mAP using C:\Users\angel\runs\detect\val5\predictions.json and C:\Users\angel\datasets\coco\annotations\instances_val20

In [ ]:
# Define classes based on the validation call: 0 is person, 67 is cell phone
class_names = ['Person', 'Cell Phone']

print("Validation Metrics Per Class:")
print("-" * 30)
for i, name in enumerate(class_names):
    precision = metrics.box.p[i]
    recall = metrics.box.r[i]
    print(f"{name:<12} | Precision: {precision:.4f} | Recall: {recall:.4f}")

print("-" * 30)
print(f"Overall mAP50-95: {metrics.box.map:.4f}")

Validation Metrics Per Class:
------------------------------
Person       | Precision: 0.7741 | Recall: 0.6610
Cell Phone   | Precision: 0.5753 | Recall: 0.3464
------------------------------
Overall mAP50-95: 0.3927


### Confusion Matrix

In [ ]:
import numpy as np

if hasattr(metrics, 'confusion_matrix'):
    # Map COCO indices: 0 is Person, 67 is Cell Phone    indices = [0, 67]
    conf_matrix = metrics.confusion_matrix.matrix[indices][:, indices]

    print("Confusion Matrix (Text Format)")
    print("-" * 40)
    print(f"{'':<15} | {'Pred Person':<12} | {'Pred Phone':<12}")
    print("-" * 40)
    print(f"{'Actual Person':<15} | {int(conf_matrix[0][0]):<12} | {int(conf_matrix[0][1]):<12}")
    print(f"{'Actual Phone':<15} | {int(conf_matrix[1][0]):<12} | {int(conf_matrix[1][1]):<12}")
    print("-" * 40)
else:
    print("Standard confusion matrix data not found in metrics.")

Confusion Matrix (Text Format)
----------------------------------------
                | Pred Person  | Pred Phone  
----------------------------------------
Actual Person   | 7291         | 0           
Actual Phone    | 0            | 89          
----------------------------------------
